# Our Goal : 

Build a brochure for a company that could be used for prospective clients, investors and potential recruiters

We will be provided with company name and website

In [1]:
# Imports
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('API_KEY')
base_url=os.getenv('BASE_URL')

if api_key and api_key.startswith('gsk_') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'llama-3.3-70b-versatile'
openai = OpenAI(base_url=base_url, api_key=api_key)

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

### Our LLM figures out which links are relevent to us

In [5]:
link_system_prompt="""
You are provided with a list of links found on the webpage
You are able to decide which of these would be most relevent to include in the brochure about the company
such as links of About page, Company page or the Jobs/Careers page
You should respond in JSON as in this example:

{
    "links": [
        {"type":"about page","url":"https://full_url_of_about_page"}
        {"type":"careers page","url":"https://full_url_of_careers_page"}
    ]
}

"""

In [7]:
def get_links_user_prompt(url):
    user_prompt=f"""
Here is a list of links on the wesite {url} -
Please decide which of these are relevent web links for a brochureq about the company
respond with the full https:// url in JSON format as specified.
Do not include Terms/Service, Privacy and email links.

Links (Some might be relevant links) - 

"""
    
    links=fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [8]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is a list of links on the wesite https://edwarddonner.com -
Please decide which of these are relevent web links for a brochureq about the company
respond with the full https:// url in JSON format as specified.
Do not include Terms/Service, Privacy and email links.

Links (Some might be relevant links) - 

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.

In [32]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by using {MODEL}")
    response=openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role":"system","content":link_system_prompt},
            {"role":"user","content":get_links_user_prompt(url)}
        ],
        response_format={"type":"json_object"}
    )

    result=response.choices[0].message.content
    links=json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [24]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by using llama-3.3-70b-versatile


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog/posts page', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'personal linkedin page',
   'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'personal twitter page', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'personal facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [33]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by using llama-3.3-70b-versatile
Found 6 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page', 'url': 'https://huggingface.co/'},
  {'type': 'learn/documentation', 'url': 'https://huggingface.co/docs'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'github repository', 'url': 'https://github.com/huggingface'}]}

### Make the Borchure

In [44]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [45]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by using llama-3.3-70b-versatile
Found 5 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/diffusiongemma-26B-A4B-it
Updated
3 days ago
•
92.1k
•
699
moonshotai/Kimi-K2.7-Code
Updated
1 day ago
•
1.69k
•
502
nvidia/LocateAnything-3B
Updated
1 day ago
•
69.4k
•
1.96k
MiniMaxAI/MiniMax-M3
Updated
about 20 hours ago
•
1.03k
•
410
CohereLabs/North-Mini-Code-1.0
Updated


In [46]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [47]:
def get_brochure_user_prompt(company_name,url):
    user_prompt=f"""
You are looking at a company called {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt+=fetch_page_and_all_relevant_links(url)
    user_prompt=user_prompt[:5000]
    return user_prompt

In [48]:
get_brochure_user_prompt("Hugging Face","https://huggingface.co")

Selecting relevant links for https://huggingface.co by using llama-3.3-70b-versatile
Found 6 relevant links


"\nYou are looking at a company called Hugging Face\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/diffusiongemma-26B-A4B-it\nUpdated\n3 days ago\n•\n92.1k\n•\n699\nmoonshotai/Kimi-K2.7-Code\nUpdated\n1 day ago\n•\n1.6

In [49]:
def create_brochure(company_name,url):
    response=openai.chat.completions.create(
        model=MODEL,
        messages=[
            {'role':"system", 'content':brochure_system_prompt},
            {'role':'user', 'content':get_brochure_user_prompt(company_name,url)}
        ]
    )
    result=response.choices[0].message.content
    display(Markdown(result))

In [56]:
create_brochure("Hugging Face","https://huggingface.co")

Selecting relevant links for https://huggingface.co by using llama-3.3-70b-versatile
Found 8 relevant links


# Hugging Face: Empowering the AI Community
Hugging Face is a pioneering platform that brings together the machine learning community to collaborate, create, and innovate. As a hub for AI development, we provide a space for individuals and organizations to share knowledge, models, datasets, and applications, driving progress in the field.

## Our Mission
Our mission is to build a future where AI benefits everyone. We strive to make machine learning more accessible, collaborative, and community-driven. By providing a platform for developers, researchers, and organizations to work together, we aim to accelerate innovation and push the boundaries of what is possible with AI.

## Company Culture
At Hugging Face, we value collaboration, openness, and inclusivity. We believe that by working together and sharing knowledge, we can achieve great things. Our community is at the heart of everything we do, and we strive to create a welcoming and supportive environment for all members.

## Our Community
Our community is comprised of talented individuals and organizations from around the world. We have over 2 million models, 1 million applications, and 500,000 datasets available on our platform, making it one of the largest and most diverse AI communities in the world. Our community members include researchers, developers, and industry leaders, all working together to advance the field of AI.

## Careers at Hugging Face
We're always looking for talented individuals to join our team. If you're passionate about AI, machine learning, and community-building, we'd love to hear from you. Our careers page offers a range of opportunities, from engineering and research to marketing and sales.

## Customers and Partners
Our platform is used by a wide range of customers, from startups to large enterprises, and from research institutions to government agencies. We partner with organizations to provide customized solutions, support, and training, helping them to achieve their AI goals.

## What We Offer
* **Models**: Browse and contribute to our vast library of pre-trained models, including state-of-the-art architectures and specialized models for specific tasks.
* **Datasets**: Discover and share datasets, from small-scale experiments to large-scale industrial applications.
* **Spaces**: Collaborate on applications, from research projects to production-ready solutions.
* **Buckets**: Store and manage your data, models, and applications in our secure and scalable storage solution.
* **Hugging Face PRO**: Get access to premium features, support, and services, designed for large-scale enterprises and organizations.

## Join the Hugging Face Community
Whether you're a seasoned researcher, a budding developer, or simply interested in AI, we invite you to join our community. Create an account, explore our platform, and start contributing to the future of AI today.

### Streaming Results (Typewriter Animation)

In [63]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Have a short title "+brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [ ]:
stream_brochure("Hugging Face","https://huggingface.co")

Selecting relevant links for https://huggingface.co by using llama-3.3-70b-versatile
Found 7 relevant links
